# Modèle de prévision - hausse de loyer 2026 (modèle D)

**Cible (inchangée) :** croissance annualisée du **loyer effectif** à unité constante, médiane de tous les baux débutant
dans l'année (renouvellements et relocations, six immeubles).

**Modèle :** un gradient boosting (scikit-learn) entraîné **au niveau des paires de baux**. Il suit littéralement la
définition : il prédit la hausse de **chaque bail** de l'année à partir de son bail précédent, puis prend la **médiane**.
On ne retient un changement que s'il améliore le modèle **à la fois** sur la médiane annuelle et sur la prévision de
chaque bail.

### Ce qui a changé par rapport au modèle D initial

| Changement | Pourquoi |
|---|---|
| nouvelle variable **`asking_gap_pct`** : dernier loyer affiché du logement (`equinoxe_asking_history`) par rapport au loyer contractuel du bail précédent | une information connue avant le nouveau bail, que le modèle D initial n'utilisait pas : un loyer affiché au-dessus du loyer payé annonce une hausse plus forte |
| backtest étendu à **2021-2025** (au lieu de 2023-2025) | 3 années ne suffisent pas à départager des variantes : un seul écart change la conclusion |
| deuxième critère : **erreur médiane par bail** (environ 2 600 baux hors période) | il repose sur des milliers de baux et non sur 5 médianes; il sert à vérifier qu'un gain annuel n'est pas dû au hasard |
| prévision 2026 accompagnée d'une **fourchette** | un seul chiffre cache l'incertitude, qui est grande |

### Comment nos analyses se relient dans le modèle

| Notebook | Ce qu'il a établi | Rôle dans le modèle |
|---|---|---|
| `equinoxe-2026` (section 1) | la définition de la hausse | la **cible** : médiane des hausses de chaque bail |
| `exploration` / section 2 | `sRentEffective` est fiable; clé d'unité; colonnes | les **données** utilisées sont valides |
| `unite_constante` | apparier, filtrer, annualiser | les **paires** sur lesquelles le modèle apprend |
| `concessions` | effectif = contractuel - hausse du rabais | la variable **rabais du bail précédent** |
| `renouvellements_relocations` | renouvellements et relocations suivent des règles différentes; The Met relève de l'Ontario | les variables **type de bail** et **taux réglementaire par province** |
| `donnees_publiques` + taux réglementaires | SCHL, TAL, Ontario | les **variables externes** testées |

## 0. Configuration et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))   # évite un avertissement de joblib sous Windows
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

LEASES_PATH = "private_data/equinoxe_lease_history.csv"
CMHC_PATH = "public_data/cmhc_market_data.csv"
REGULATORY_RATES_PATH = "public_data/regulatory_rates.csv"
ASKING_PATH = "private_data/equinoxe_asking_history.csv"

UNIT_KEY = ["sPropCode", "sUnitCode"]
MIN_GAP_YEARS = 0.5
MAX_GAP_YEARS = 2.5
DAYS_PER_YEAR = 365.25
PCT = 100
RENEWAL, TURNOVER = 1, 0
ONTARIO = "Ontario"
MONTREAL_MARKET = "Montréal"

FIRST_HISTORY_YEAR = 2019
BACKTEST_YEARS = [2021, 2022, 2023, 2024, 2025]
INITIAL_BACKTEST_YEARS = [2023, 2024, 2025]   # période de backtest du modèle D initial, gardée pour comparaison
FORECAST_YEAR = 2026
OTTAWA_MARKET = "Ottawa — partie Ontario"

# Modèle D (gradient boosting au niveau des paires)
UNIT_FEATURES = ["sBeds", "sSqft", "sFloor", "gap_years", "is_renewal"]
GBM_PARAMS = {
    "loss": "absolute_error",   # erreur absolue : le modèle vise la médiane, comme notre définition
    "max_iter": 200,
    "learning_rate": 0.05,
    "max_depth": 3,             # arbres peu profonds : peu de données par année
    "min_samples_leaf": 40,     # chaque feuille repose sur au moins 40 paires
    "random_state": 0,
}
MODEL_D_PATH = Path("models/modele_d_gbm.joblib")

LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",
    "sRenewal": "is_renewal",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sTermMonths": "lease_term_months",
}

leases = pd.read_csv(LEASES_PATH, parse_dates=["sLeaseFrom", "sLeaseTo"]).rename(columns=LEASE_COLUMN_RENAMES)
leases["lease_start_year"] = leases["lease_start"].dt.year
cmhc = pd.read_csv(CMHC_PATH)
regulatory_rates = pd.read_csv(REGULATORY_RATES_PATH).set_index("year")
asking = pd.read_csv(ASKING_PATH)
asking["ask_month"] = pd.PeriodIndex(asking["sMonth"], freq="M").to_timestamp()
print(f"{len(leases):,} baux | {len(asking):,} loyers affichés ({asking['sMonth'].min()} à {asking['sMonth'].max()}) | SCHL : {cmhc['year'].min()}-{cmhc['year'].max()} | taux réglementaires : "
      f"{regulatory_rates.index.min()}-{regulatory_rates.index.max()}")

## 1. Construire les paires de baux et l'historique de la cible

On reprend la méthode de `unite_constante` : chaque bail est comparé au bail précédent de la même unité, les écarts non
comparables sont filtrés, la variation est annualisée. Pour chaque paire, on calcule aussi la **hausse du rabais**
(rabais du nouveau bail - rabais du bail précédent, en points), mécanisme établi dans `concessions`.

Chaque paire est rangée dans un **segment** :
- `renewal_QC` : renouvellement dans un immeuble québécois (encadré par le TAL);
- `renewal_ON` : renouvellement à The Met (Ontario, probablement exempté du plafond);
- `turnover` : relocation (loyer de marché, sans plafond dans les deux provinces).

In [ ]:
def build_comparable_pairs(lease_df):
    """Paires de baux consécutifs d'une même unité, filtrées, avec croissances annualisées et hausse du rabais."""
    ordered = lease_df.sort_values(UNIT_KEY + ["lease_start"]).copy()
    previous = (ordered.groupby(UNIT_KEY)[["contract_rent", "effective_rent", "lease_start"]]
                .shift(1).add_prefix("prev_"))
    pairs = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    pairs["gap_years"] = (pairs["lease_start"] - pairs["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    is_comparable = pairs["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    pairs = pairs[is_comparable].copy()

    for rent in ["contract_rent", "effective_rent"]:
        ratio = pairs[rent] / pairs[f"prev_{rent}"]
        pairs[f"{rent}_growth_pct"] = (ratio ** (1 / pairs["gap_years"]) - 1) * PCT

    discount_now = (1 - pairs["effective_rent"] / pairs["contract_rent"]) * PCT
    discount_before = (1 - pairs["prev_effective_rent"] / pairs["prev_contract_rent"]) * PCT
    pairs["discount_change_pts"] = discount_now - discount_before
    pairs["prev_discount_pct"] = discount_before

    is_ontario = pairs["sState"] == ONTARIO
    pairs["segment"] = np.select(
        [pairs["is_renewal"].eq(RENEWAL) & ~is_ontario, pairs["is_renewal"].eq(RENEWAL) & is_ontario],
        ["renewal_QC", "renewal_ON"], default="turnover")
    return pairs


pairs = build_comparable_pairs(leases)
pairs = pairs[pairs["lease_start_year"] >= FIRST_HISTORY_YEAR]
print(f"{len(pairs):,} paires comparables depuis {FIRST_HISTORY_YEAR}")

In [ ]:
yearly_target = pairs.groupby("lease_start_year").agg(
    pairs=("effective_rent_growth_pct", "size"),
    effective_growth=("effective_rent_growth_pct", "median"),
    contract_growth=("contract_rent_growth_pct", "median"),
    discount_change=("discount_change_pts", "median"))

yearly_segments = pairs.groupby(["lease_start_year", "segment"]).agg(
    pairs=("effective_rent_growth_pct", "size"),
    contract_growth=("contract_rent_growth_pct", "median"),
    discount_change=("discount_change_pts", "median")).unstack("segment")

print("Cible et composantes, tous baux :")
print(yearly_target.round(2), end="\n\n")
print("Par segment :")
print(yearly_segments.round(2))

**Lecture :** la cible (`effective_growth`) oscille entre 1,6 % et 4,6 % sur 2019-2025, avec un creux en 2023-2024.
L'écart entre contractuel et effectif suit la hausse du rabais : nul jusqu'en 2023, puis 1,2 point en 2024 et 3,0 points en 2025.
Les renouvellements à The Met n'existent qu'à partir de 2024 : pour les années antérieures, il n'y a pas d'historique ontarien.

## 2. Relations avec les données externes

Avant de les utiliser dans un modèle, on vérifie si les variables externes sont **réellement liées** à nos composantes.
Avec 5 à 7 années, ces relations sont **exploratoires** : elles orientent le modèle, elles ne prouvent rien.

### 2a. Renouvellements québécois et taux du TAL

Si les renouvellements suivaient le TAL, la « prime » (hausse observée - taux du TAL) serait stable d'une année à l'autre.

In [ ]:
renewal_vs_tal = pd.DataFrame({
    "renewal_QC_contract_growth": yearly_segments[("contract_growth", "renewal_QC")],
    "tal_rate": regulatory_rates["tal_unheated_pct"],
}).dropna()
renewal_vs_tal["premium_over_tal"] = renewal_vs_tal["renewal_QC_contract_growth"] - renewal_vs_tal["tal_rate"]
print(renewal_vs_tal.round(2))
print(f"\nCorrélation hausse des renouvellements QC / taux TAL : {renewal_vs_tal.iloc[:, :2].corr().iloc[0, 1]:.2f}")

**Lecture :** la relation est faible et la prime est **instable** : environ +4 points en 2021-2022 (TAL très bas,
Équinoxe augmente bien au-dessus), puis proche de 0 depuis 2023 (+0,3, -1,1, +0,5). Depuis que le TAL est élevé, les
renouvellements québécois suivent à peu près le taux du TAL; avant, ils l'ignoraient. Le taux du TAL n'est qu'une
recommandation : il ne s'impose que si le locataire conteste. On ne l'impose donc pas comme ancrage :
le taux réglementaire entre dans le modèle comme **une variable parmi d'autres**, et le modèle apprend lui-même son poids.

### 2b. Hausse du rabais et inoccupation du marché (SCHL)

C'est l'hypothèse « vacance → concessions » testée sans succès avec les données internes (section 2g du notebook final),
reprise ici avec le taux d'inoccupation **du marché de Montréal** publié par la SCHL (enquête d'octobre).

In [ ]:
montreal = cmhc[cmhc["market"] == MONTREAL_MARKET].set_index("year")
discount_vs_vacancy = pd.DataFrame({
    "discount_change": yearly_target["discount_change"],
    "vacancy_rate": montreal["vacancy_rate"],
    "vacancy_change": montreal["vacancy_rate"].diff(),
}).dropna()
print(discount_vs_vacancy.round(2))
print(f"\nCorrélation hausse du rabais / variation de l'inoccupation (même année) : "
      f"{discount_vs_vacancy[['discount_change', 'vacancy_change']].corr().iloc[0, 1]:.2f}  (n = {len(discount_vs_vacancy)})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(discount_vs_vacancy["vacancy_change"], discount_vs_vacancy["discount_change"])
for year, row in discount_vs_vacancy.iterrows():
    ax.annotate(str(year), (row["vacancy_change"], row["discount_change"]), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("variation du taux d'inoccupation, Montréal (points, SCHL)")
ax.set_ylabel("hausse médiane du rabais (points)")
ax.set_title("Quand l'inoccupation du marché monte, les rabais augmentent")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** l'inoccupation à Montréal baisse jusqu'en 2023 (1,5 %, marché très serré), puis remonte (2,1 % en 2024,
2,9 % en 2025). Les rabais d'Équinoxe suivent : stables quand le marché se resserre, en forte hausse quand il se détend
(corrélation d'environ 0,8). C'est cohérent avec l'hypothèse de départ : **les concessions répondent au marché**.

**Prudence :** 4 points seulement, et la relation est **contemporaine** - l'inoccupation d'octobre de l'année *t* n'est
connue qu'à la fin de *t*. Elle ne peut donc pas servir directement à prévoir *t*; Dans le modèle, on teste l'inoccupation
d'octobre de l'année **précédente**, la seule connue au moment de prévoir (section 5).

## 3. Le modèle : gradient boosting au niveau des paires (scikit-learn)

Le modèle apprend la hausse d'une paire à partir de ce qu'on sait **avant** le nouveau
bail, prédit chaque bail de l'année, et prend la médiane. Il apprend sur environ 3 000 paires, et non sur 7 années.

**Variables (toutes connues avant le début du nouveau bail) :**

| Variable | Pourquoi |
|---|---|
| immeuble (indicatrices), chambres, superficie, étage | le produit loué |
| `gap_years` | durée écoulée depuis le bail précédent |
| `prev_discount_pct` | rabais du bail précédent : un rabais élevé peut reculer au bail suivant, et inversement |
| `regulatory_rate` | taux du TAL (Québec) ou ligne directrice (Ontario) de l'année, publiés avant l'année |
| `is_renewal` | renouvellement ou relocation |
| **`asking_gap_pct`** (nouveau) | écart (en %, log) entre le **dernier loyer affiché** du logement avant le 1er janvier de l'année et le loyer contractuel du bail précédent |

**Pourquoi `asking_gap_pct` :** `equinoxe_asking_history` donne, mois par mois, le loyer affiché de chaque logement.
Quand le propriétaire affiche un logement plus cher que ce que paie le locataire, le bail suivant tend à monter davantage.
On ne garde que les affichages **antérieurs au 1er janvier** de l'année prévue : la variable est donc connue au moment de
prévoir, en backtest comme pour 2026 (l'historique va jusqu'en décembre 2025).

**Le type de bail n'est pas connu d'avance** : chaque bail est prédit comme renouvellement et comme relocation, pondéré
par la part historique de renouvellements de son immeuble (inchangé).

In [ ]:
BUILDINGS = sorted(leases["sBuilding"].unique())
SCHL_VACANCY = cmhc.set_index(["market", "year"])["vacancy_rate"]


def regulatory_rate_for(df):
    """Taux réglementaire de l'année du nouveau bail : TAL au Québec, ligne directrice en Ontario."""
    years = df["lease_start_year"]
    tal = years.map(regulatory_rates["tal_unheated_pct"])
    ontario = years.map(regulatory_rates["ontario_guideline_pct"])
    return np.where(df["sState"] == ONTARIO, ontario, tal)


def previous_october_vacancy(df):
    """Taux d'inoccupation SCHL d'octobre de l'année précédente, pour le marché de l'immeuble."""
    markets = np.where(df["sState"] == ONTARIO, OTTAWA_MARKET, MONTREAL_MARKET)
    return [SCHL_VACANCY.get((market, year - 1), np.nan) for market, year in zip(markets, df["lease_start_year"])]


def last_asking_rent_before_year(df):
    """Dernier loyer affiché du logement avant le 1er janvier de l'année du nouveau bail (connu au moment de prévoir)."""
    query = pd.DataFrame({
        "row": np.arange(len(df)),
        "sPropCode": df["sPropCode"].to_numpy(),
        "sUnitCode": df["sUnitCode"].to_numpy(),
        "cutoff": pd.to_datetime(df["lease_start_year"].astype(str) + "-01-01").to_numpy(),
    }).sort_values("cutoff")
    history = asking[UNIT_KEY + ["ask_month", "sAskingRent"]].sort_values("ask_month")
    matched = pd.merge_asof(query, history, left_on="cutoff", right_on="ask_month", by=UNIT_KEY,
                            allow_exact_matches=False)
    return matched.sort_values("row")["sAskingRent"].to_numpy()


def asking_gap_pct(df):
    """Écart (%, log) entre le dernier loyer affiché et le loyer contractuel du bail précédent."""
    return np.log(last_asking_rent_before_year(df) / df["prev_contract_rent"].to_numpy()) * PCT


def feature_matrix(df, use_prev_discount=True, use_regulatory=True, use_vacancy=False, use_asking_gap=True):
    features = df[UNIT_FEATURES].astype(float).copy()
    if use_prev_discount:
        features["prev_discount_pct"] = df["prev_discount_pct"]
    if use_regulatory:
        features["regulatory_rate"] = regulatory_rate_for(df)
    if use_vacancy:
        features["previous_october_vacancy"] = previous_october_vacancy(df)
    if use_asking_gap:
        features["asking_gap_pct"] = asking_gap_pct(df)
    for building in BUILDINGS:
        features[f"building_{building}"] = (df["sBuilding"] == building).astype(float)
    return features


def weighted_median(values, weights):
    order = np.argsort(values)
    sorted_values, cumulative_weights = values[order], np.cumsum(weights[order])
    return sorted_values[np.searchsorted(cumulative_weights, cumulative_weights[-1] / 2)]


def predict_both_lease_types(model, population, training_pairs, feature_options):
    """Prédit chaque bail comme renouvellement et comme relocation, pondéré par la part de renouvellements de l'immeuble."""
    renewal_share = (population["sBuilding"].map(training_pairs.groupby("sBuilding")["is_renewal"].mean())
                     .fillna(training_pairs["is_renewal"].mean()).to_numpy())
    scenarios = []
    for status, weight in [(RENEWAL, renewal_share), (TURNOVER, 1 - renewal_share)]:
        scenario = population.assign(is_renewal=status)
        scenario["predicted_growth"] = predict_pairs(model, scenario, feature_options)
        scenario["weight"] = weight
        scenarios.append(scenario)
    return pd.concat(scenarios)


def fit_pair_model(training_pairs, feature_options):
    features = feature_matrix(training_pairs, **feature_options)
    features = features.loc[:, features.notna().any()]   # ex. SCHL absente avant 2021 : colonne vide, ignorée
    model = HistGradientBoostingRegressor(**GBM_PARAMS)
    model.fit(features, training_pairs["effective_rent_growth_pct"])
    return model


def predict_pairs(model, df, feature_options):
    return model.predict(feature_matrix(df, **feature_options)[model.feature_names_in_])

### Population de baux à prévoir

- **Pour une année de backtest** : les baux qui ont réellement débuté cette année-là. On n'utilise que les informations de leur
  **bail précédent** (rabais, durée écoulée) et du logement, connues avant le début du nouveau bail.
- **Pour 2026** : le dernier bail connu de chaque logement se termine à une date connue (`lease_end`). Le prochain bail débute
  le lendemain. On retient les logements dont ce prochain bail débute en 2026, avec les mêmes filtres d'écart que pour l'historique.

In [ ]:
last_known_leases = leases.sort_values("lease_start").groupby(UNIT_KEY).tail(1).copy()
next_lease_start = last_known_leases["lease_end"] + pd.Timedelta(days=1)
population_2026 = last_known_leases.assign(
    lease_start_year=next_lease_start.dt.year,
    gap_years=(next_lease_start - last_known_leases["lease_start"]).dt.days / DAYS_PER_YEAR,
    prev_contract_rent=last_known_leases["contract_rent"],
    prev_discount_pct=(1 - last_known_leases["effective_rent"] / last_known_leases["contract_rent"]) * PCT)
population_2026 = population_2026[
    (population_2026["lease_start_year"] == FORECAST_YEAR)
    & population_2026["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")]

print(f"{len(population_2026):,} logements dont le prochain bail débute en {FORECAST_YEAR}")
print(population_2026["sBuilding"].value_counts().to_string())
print(f"\nRabais moyen du bail précédent : population 2026 = {population_2026['prev_discount_pct'].mean():.1f} %, "
      f"baux de {FORECAST_YEAR - 1} = {pairs.loc[pairs['lease_start_year'] == FORECAST_YEAR - 1, 'prev_discount_pct'].mean():.1f} %")


def population_for(year):
    return population_2026 if year == FORECAST_YEAR else pairs[pairs["lease_start_year"] == year]


RETAINED_FEATURES = {"use_prev_discount": True, "use_regulatory": True, "use_vacancy": False, "use_asking_gap": True}
INITIAL_FEATURES = {**RETAINED_FEATURES, "use_asking_gap": False}   # modèle D initial, sans loyer affiché


def forecast_year(year, feature_options=RETAINED_FEATURES):
    """Entraîne sur les paires antérieures à `year`, prédit chaque bail de `year`, renvoie la médiane pondérée (%)."""
    training_pairs = pairs[pairs["lease_start_year"] < year]
    model = fit_pair_model(training_pairs, feature_options)
    predictions = predict_both_lease_types(model, population_for(year), training_pairs, feature_options)
    return weighted_median(predictions["predicted_growth"].to_numpy(), predictions["weight"].to_numpy())


print(f"\nLoyer affiché disponible : {np.isfinite(asking_gap_pct(population_2026)).mean():.0%} des logements de "
      f"{FORECAST_YEAR}, {np.isfinite(asking_gap_pct(pairs)).mean():.0%} des paires historiques")
print(f"asking_gap_pct médian : population {FORECAST_YEAR} = {np.nanmedian(asking_gap_pct(population_2026)):.2f} %, "
      f"paires historiques = {np.nanmedian(asking_gap_pct(pairs)):.2f} %")

## 4. Backtest

Règle (inchangée) : pour prévoir l'année *t*, le modèle n'utilise que ce qui était **connu au début de *t*** :

- il est entraîné uniquement sur les paires dont le nouveau bail a débuté **avant** *t*;
- pour les baux de *t*, il ne voit que le logement, le **bail précédent** (rabais, durée écoulée) et les loyers affichés
  **avant le 1er janvier de *t***, pas le nouveau loyer;
- le type de bail (renouvellement ou relocation) n'est pas supposé connu : les deux cas sont pondérés (section 3);
- les taux réglementaires de *t* sont connus (publiés avant le début de l'année); la SCHL ne l'est que jusqu'à octobre *t* - 1.

**Deux critères :**
- **MAE annuelle** : erreur absolue moyenne de la médiane prévue (prévision - réel, en points), sur **2021-2025**.
  On affiche aussi 2023-2025, la période de backtest du modèle D initial.
- **Erreur par bail** : pour chaque année de backtest, erreur absolue médiane entre la hausse prévue et la hausse réelle
  de chaque bail (type de bail réel), puis moyenne sur les années. Ce critère repose sur environ 2 600 baux : il départage
  les variantes beaucoup plus sûrement que 5 médianes annuelles.

In [ ]:
def backtest(target_year, feature_options=RETAINED_FEATURES):
    """Prévoit target_year avec l'information disponible à la fin de target_year - 1, puis compare au réel."""
    forecast = forecast_year(target_year, feature_options)
    actual = yearly_target.loc[target_year, "effective_growth"]
    return {"forecast": forecast, "actual": actual, "error": forecast - actual}


def lease_level_error(feature_options=RETAINED_FEATURES):
    """Erreur absolue médiane par bail, hors période, moyennée sur les années de backtest (points)."""
    yearly_errors = []
    for year in BACKTEST_YEARS:
        training_pairs = pairs[pairs["lease_start_year"] < year]
        test_pairs = pairs[pairs["lease_start_year"] == year]
        model = fit_pair_model(training_pairs, feature_options)
        predicted = predict_pairs(model, test_pairs, feature_options)
        yearly_errors.append(np.median(np.abs(predicted - test_pairs["effective_rent_growth_pct"])))
    return np.mean(yearly_errors)


backtest_results = pd.DataFrame({year: backtest(year) for year in BACKTEST_YEARS}).T
backtest_initial = pd.DataFrame({year: backtest(year, INITIAL_FEATURES) for year in BACKTEST_YEARS}).T
backtest_results["forecast_initial"] = backtest_initial["forecast"]
model_mae = backtest_results["error"].abs().mean()

print(backtest_results.round(2), end="\n\n")
comparison = pd.DataFrame({
    "D initial": [backtest_initial["error"].abs().mean(), backtest_initial.loc[INITIAL_BACKTEST_YEARS, "error"].abs().mean(),
                  lease_level_error(INITIAL_FEATURES)],
    "D retenu": [model_mae, backtest_results.loc[INITIAL_BACKTEST_YEARS, "error"].abs().mean(), lease_level_error()],
}, index=["MAE 2021-2025", "MAE 2023-2025", "erreur par bail"])
print(comparison.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(yearly_target.index, yearly_target["effective_growth"], "k-o", lw=2.5, label="réel (cible)")
ax.plot(backtest_results.index, backtest_results["forecast_initial"], ":^", color="grey", lw=1.5, label="prévision, modèle D initial")
ax.plot(backtest_results.index, backtest_results["forecast"], "--s", color="tab:blue", lw=2, label="prévision, modèle retenu")
ax.axvspan(min(BACKTEST_YEARS) - 0.5, max(BACKTEST_YEARS) + 0.5, color="grey", alpha=0.1, label="période de backtest")
ax.set_xlabel("année")
ax.set_ylabel("croissance du loyer effectif, même unité (%)")
ax.set_title("Backtest : prévisions faites avec l'information disponible l'année précédente")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :**

- **Le loyer affiché améliore le modèle sur les deux critères.** Erreur par bail : 2,88 points contre 2,95 pour le modèle D initial. Sur la période de backtest initiale (2023-2025), la MAE annuelle passe de 1,25 à 1,11 point. Sur 2021-2025, elle reste à 1,04 point : le gain de 2023-2024 est compensé par 2022, moins bien prévu.
- **La prévision 2026 ne bouge presque pas** (3,58 % contre 3,55 %) : l'amélioration ne repose pas sur un changement d'hypothèse pour 2026.
- **2023 reste la grande erreur** (4,3 % prévu pour 2,0 % réel), comme dans le modèle D initial. Le ralentissement de 2023 est un choc d'année : aucune variable connue fin 2022 ne l'annonçait. Le modèle apprend la hausse d'un bail *selon ses caractéristiques*, mais ne peut pas deviner qu'une année entière sera plus faible.
- Pour comparaison, reprendre la hausse de l'année précédente donne une MAE de 1,24 point sur 2021-2025 (1,54 sur 2023-2025); la moyenne des 3 dernières années, 1,43 (1,36).

## 5. Quelles variables comptent?

On refait le backtest et la prévision 2026 en retirant ou en ajoutant **un groupe de variables à la fois**, à partir du
modèle retenu. Chaque ligne est jugée sur les deux critères.

In [ ]:
FEATURE_VARIANTS = {
    "retenu (D initial + loyer affiché)": RETAINED_FEATURES,
    "D initial (sans loyer affiché)": INITIAL_FEATURES,
    "+ inoccupation SCHL": {**RETAINED_FEATURES, "use_vacancy": True},
    "sans rabais précédent": {**RETAINED_FEATURES, "use_prev_discount": False},
    "sans taux réglementaire": {**RETAINED_FEATURES, "use_regulatory": False},
    "logement + loyer affiché seulement": {**RETAINED_FEATURES, "use_prev_discount": False, "use_regulatory": False},
}

variant_rows = {}
for variant_name, options in FEATURE_VARIANTS.items():
    row = {year: backtest(year, options)["forecast"] for year in BACKTEST_YEARS}
    errors = {year: row[year] - yearly_target.loc[year, "effective_growth"] for year in BACKTEST_YEARS}
    row["MAE 21-25"] = np.mean(np.abs(list(errors.values())))
    row["MAE 23-25"] = np.mean([abs(errors[year]) for year in INITIAL_BACKTEST_YEARS])
    row["erreur/bail"] = lease_level_error(options)
    row[FORECAST_YEAR] = forecast_year(FORECAST_YEAR, options)
    variant_rows[variant_name] = row
variants = pd.DataFrame(variant_rows).T
variants.round(2)

**Lecture :**

- **Le rabais du bail précédent reste la variable la plus utile** : sans elle, l'erreur par bail passe de 2,88 à 3,26 points et la MAE annuelle de 1,04 à 1,34. C'est aussi elle qui porte la prévision 2026 : sans elle, 2,3 % au lieu de 3,6 %. Les logements de 2026 partent d'un rabais élevé (9,0 % en moyenne) : le modèle a appris qu'après un rabais élevé, le loyer effectif du bail suivant monte davantage.
- **Le taux réglementaire** n'aide pas la MAE annuelle (0,95 sans lui), mais il améliore la prévision de chaque bail (3,02 sans lui contre 2,88). Comme il ne prend qu'une valeur par année et par province, les arbres s'en servent en partie pour **reconnaître l'année**. Le retirer fait monter 2026 à 4,3 % : la prévision dépend donc aussi de cette variable. On la garde, puisqu'elle est utile bail par bail et qu'elle a un sens économique.
- **L'inoccupation SCHL** n'apporte presque rien (2,89 par bail, MAE 1,01 contre 1,04) mais fait bondir 2026 à 5,0 % : l'inoccupation d'octobre 2025 (2,9 % à Montréal) dépasse tout ce que le modèle a vu, avec seulement 4 années pour en apprendre l'effet. Gain négligeable, extrapolation forte : on ne la retient pas, comme dans le modèle D initial.
- **Testé hors notebook, sans gain robuste** (gain sur un critère, perte sur l'autre, ou écarts de quelques centièmes) : rabais précédent mesuré par rapport à la moyenne de son année, position du loyer dans l'immeuble, loyer affiché médian de l'immeuble, élan des loyers affichés de l'année précédente, hausse de l'année précédente comme variable, pondération des années récentes, profondeur des arbres, taille des feuilles, régularisation, moyenne avec la règle « hausse de l'an dernier », décomposition « croissance contractuelle × hypothèse de rabais », et moyenne de plusieurs variantes.

## 6. Prévision 2026

On applique le modèle avec tout l'historique jusqu'à 2025, les taux réglementaires 2026 (TAL : 3,1 %; Ontario : 2,1 %) et
les loyers affichés jusqu'en décembre 2025. La prévision par immeuble et par nombre de chambres répond au bonus des
consignes. Le modèle final est entraîné sur toutes les paires jusqu'en 2025, puis sauvegardé (livrable « modèle entraîné »).

**Fourchette**, établie de deux façons complémentaires :
- **erreur typique du backtest** : la prévision ± la MAE 2021-2025;
- **sensibilité aux choix de modélisation** : l'étendue des prévisions 2026 des variantes de la section 5 dont l'erreur
  par bail reste à 0,1 point de celle du modèle retenu.

In [ ]:
def estimate_2026(feature_options=RETAINED_FEATURES):
    """Hausse 2026 du loyer effectif à unité constante (%), selon notre définition."""
    return forecast_year(FORECAST_YEAR, feature_options)


forecast_2026 = estimate_2026()
retained_lease_error = variants.loc["retenu (D initial + loyer affiché)", "erreur/bail"]
comparable_variants = variants[variants["erreur/bail"] <= retained_lease_error + 0.1]
print(f"Hausse 2026 prévue (loyer effectif, même unité, médiane) : {forecast_2026:.2f} %")
print(f"  erreur typique du backtest : {forecast_2026 - model_mae:.2f} % à {forecast_2026 + model_mae:.2f} %")
print(f"  variantes comparables ({len(comparable_variants)}) : {comparable_variants[FORECAST_YEAR].min():.2f} % "
      f"à {comparable_variants[FORECAST_YEAR].max():.2f} %\n")

final_training_pairs = pairs[pairs["lease_start_year"] < FORECAST_YEAR]
final_model_d = fit_pair_model(final_training_pairs, RETAINED_FEATURES)
predictions_2026 = predict_both_lease_types(final_model_d, population_2026, final_training_pairs, RETAINED_FEATURES)


def weighted_median_by(df, group_column):
    rows = {}
    for group_value, group in df.groupby(group_column):
        rows[group_value] = {
            "baux_2026": len(group) // 2,   # chaque bail apparaît deux fois (renouvellement et relocation)
            "hausse_effective_prevue_pct": weighted_median(group["predicted_growth"].to_numpy(),
                                                           group["weight"].to_numpy()),
        }
    return pd.DataFrame(rows).T


print("Par immeuble :")
print(weighted_median_by(predictions_2026, "sBuilding").round(2), end="\n\n")
print("Par nombre de chambres :")
print(weighted_median_by(predictions_2026, "sBeds").round(2))

MODEL_D_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"model": final_model_d, "features": RETAINED_FEATURES, "buildings": BUILDINGS}, MODEL_D_PATH)
print(f"\nModèle sauvegardé : {MODEL_D_PATH}")

## 7. Bilan du modèle

**Prévision 2026 : 3,6 %** de hausse du loyer effectif à unité constante (médiane des baux 2026).

**Fourchette :** environ **2,5 % à 4,6 %** selon l'erreur typique du backtest (± 1,0 point). Les variantes de qualité comparable donnent de 3,55 % à 5,0 %; la borne haute vient de l'inoccupation SCHL, écartée pour cause d'extrapolation. Par immeuble : de 3,3 % (The Met) à 3,7 % (Saint-Elzear, Westpark).

**Ce qui a été amélioré par rapport au modèle D initial :**
- une variable de plus, connue d'avance et déjà dans nos données : le **loyer affiché** du logement; meilleure sur les deux critères (erreur par bail 2,88 contre 2,95; MAE 2023-2025 1,11 contre 1,25);
- un backtest sur **5 années** au lieu de 3, et un critère **par bail** qui repose sur environ 2 600 baux : les choix de variables sont mieux appuyés;
- une **fourchette** au lieu d'un seul chiffre.

**Limites qui restent :**
1. **Les chocs d'année ne sont pas prévisibles avec ces données** (2023). Une quinzaine de variantes, de variables et de réglages, plafonnent toutes autour d'une MAE de 1,0 point : l'erreur restante vient de l'année, pas du modèle.
2. **La prévision 2026 repose sur une hypothèse sur les rabais** : partis d'un niveau élevé (9 %), ils reculeraient en partie. Si les concessions continuent d'augmenter en 2026, la hausse effective sera plus proche de 2,3 % (variante sans rabais précédent).
3. **Le taux réglementaire sert en partie d'indicateur d'année**; sans lui, la prévision monte à 4,3 %.
4. Le fichier `equinoxe_listings` (loyers affichés pour des logements disponibles jusqu'en septembre 2026) pourrait renseigner 2026 directement, mais il n'existe que pour 2025 : impossible de le valider au backtest, il n'est donc pas utilisé.